# Mini-projet : Solutions d'IA MCP + Agents performantes

Ce notebook couvre les **deux journées** du mini-projet :

- **Jour 1** — Intégrer des serveurs MCP tiers existants dans un pipeline d'orchestration piloté par un LLM (Ollama ou GroqCloud).
- **Jour 2** — Construire son propre serveur MCP avec des outils personnalisés et l'ajouter au pipeline, jusqu'à obtenir une application agentique complète (avec interface Streamlit).

**Thème choisi : « Assistant de développement »** — `git` + `filesystem` (serveurs MCP tiers) + un serveur MCP personnalisé exposant un **wrapper d'exécution de tests/linter** robuste (gestion des timeouts et des erreurs).

## Principes directeurs
- Ne pas réinventer les outils → on s'appuie d'abord sur des serveurs MCP existants et bien documentés.
- Orchestration flexible → c'est le LLM qui décide, tour après tour, quel outil appeler (pas de flux codé en dur).
- Robustesse → journalisation (`logging`) et gestion propre des erreurs (timeout, entrées invalides, sous-processus qui échoue).
- Configuration par variables d'environnement (backend LLM, endpoints).
- Reproductibilité → `clone + pip install -r requirements.txt + run`.

## Prérequis
- Python 3.10+ et notions de base.
- Un compte [Ollama](https://ollama.com/) (exécution locale) **ou** une clé API [GroqCloud](https://console.groq.com/).
- Git installé (pour cloner/exécuter des dépôts et pour le serveur MCP `git`).
- Streamlit installé (interface utilisateur, en fin de notebook).

---
## 0. Configuration générale : dépendances et variables d'environnement

In [ ]:
%pip install -qU \

  "langchain>=0.3" \

  "langgraph>=0.2" \

  "langchain-mcp-adapters==0.2.1" \

  "langchain-groq>=0.2" \

  "langchain-ollama>=0.2" \

  "mcp-server-git" \

  "python-dotenv" \

  "nest_asyncio" \

  "streamlit"

### Fichier `.env` (configuration reproductible)

Toute la configuration passe par des variables d'environnement, chargées via `python-dotenv`. Cela permet de cloner le dépôt, de créer un `.env` local, puis de lancer le notebook (ou l'app Streamlit) sans toucher au code.

Variables utilisées :
- `LLM_BACKEND` : `"groq"` ou `"ollama"` (par défaut `"ollama"`).
- `GROQ_API_KEY`, `GROQ_MODEL` (par défaut `"llama-3.3-70b-versatile"`).
- `OLLAMA_MODEL` (par défaut `"llama3.1"`), `OLLAMA_BASE_URL` (par défaut `"http://localhost:11434"`).
- `WORKDIR` : répertoire de travail utilisé par les serveurs `filesystem` et `git`.
- `TOOL_TIMEOUT_SECONDS` : délai maximal (en secondes) accordé à un appel d'outil avant abandon.

In [ ]:
%%writefile .env.example

LLM_BACKEND=ollama



# --- GroqCloud ---

GROQ_API_KEY=

GROQ_MODEL=llama-3.3-70b-versatile



# --- Ollama (local) ---

OLLAMA_MODEL=llama3.1

OLLAMA_BASE_URL=http://localhost:11434



# --- Général ---

WORKDIR=./workspace

TOOL_TIMEOUT_SECONDS=30

In [ ]:
import os

import shutil

from dotenv import load_dotenv



# Copier .env.example -> .env si absent (à personnaliser ensuite)

if not os.path.exists(".env"):

    shutil.copy(".env.example", ".env")

    print("Fichier .env créé à partir de .env.example — pensez à le compléter (clé API, backend...).")



load_dotenv(override=True)



LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()

WORKDIR = os.path.abspath(os.getenv("WORKDIR", "./workspace"))

TOOL_TIMEOUT_SECONDS = int(os.getenv("TOOL_TIMEOUT_SECONDS", "30"))



os.makedirs(WORKDIR, exist_ok=True)

print("Backend LLM :", LLM_BACKEND)

print("Répertoire de travail :", WORKDIR)

print("Timeout outil (s) :", TOOL_TIMEOUT_SECONDS)

### Journalisation (logging)

Toute erreur ou tout appel d'outil est journalisé dans `mcp_agent.log`, pour pouvoir diagnostiquer un timeout ou une entrée incorrecte sans interrompre l'application.

In [ ]:
import logging



logger = logging.getLogger("mcp_agent")

logger.setLevel(logging.INFO)

logger.handlers.clear()



file_handler = logging.FileHandler("mcp_agent.log", encoding="utf-8")

file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

logger.addHandler(file_handler)



stream_handler = logging.StreamHandler()

stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))

logger.addHandler(stream_handler)



logger.info("Session démarrée avec le backend %s", LLM_BACKEND)

### Fabrique de LLM (`get_llm`)

Selon `LLM_BACKEND`, on instancie soit `ChatGroq` (API cloud), soit `ChatOllama` (modèle local). En cas de configuration manquante (ex. clé API absente), on lève une erreur explicite plutôt qu'une exception opaque.

In [ ]:
def get_llm():

    """Retourne un LLM LangChain configuré selon la variable d'environnement LLM_BACKEND."""

    if LLM_BACKEND == "groq":

        from langchain_groq import ChatGroq

        api_key = os.getenv("GROQ_API_KEY")

        if not api_key:

            raise RuntimeError(

                "LLM_BACKEND=groq mais GROQ_API_KEY est vide. "

                "Renseignez-la dans le fichier .env."

            )

        model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

        logger.info("Initialisation de ChatGroq (modele=%s)", model)

        return ChatGroq(model=model, api_key=api_key, temperature=0)



    elif LLM_BACKEND == "ollama":

        from langchain_ollama import ChatOllama

        model = os.getenv("OLLAMA_MODEL", "llama3.1")

        base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

        logger.info("Initialisation de ChatOllama (modele=%s, base_url=%s)", model, base_url)

        return ChatOllama(model=model, base_url=base_url, temperature=0)



    else:

        raise ValueError(f"LLM_BACKEND inconnu : '{LLM_BACKEND}' (attendu: 'groq' ou 'ollama')")



llm = get_llm()

print("LLM prêt :", llm)

---
# 🗓️ Jour 1 — Intégrer des serveurs MCP tiers

**Objectif du jour :** connecter au moins deux serveurs MCP tiers bien documentés (`filesystem` et `git`) à un agent piloté par le LLM configuré ci-dessus, sans coder de flux prédéfini.

## 1.1 Vérifier Node/NPM (requis pour les serveurs MCP distribués en packages npm)

In [ ]:
!node --version

!npx --version

Si Node/NPM manquent :

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y nodejs npm

!node --version

!npx --version

## 1.2 Préparer un dépôt Git de démonstration

Le serveur `filesystem` et le serveur `git` opèrent tous deux sur `WORKDIR`.

In [ ]:
import subprocess

from pathlib import Path



def run(cmd, cwd=WORKDIR):

    """Exécute une commande shell avec gestion d'erreur et journalisation."""

    try:

        result = subprocess.run(cmd, cwd=cwd, check=True, capture_output=True, text=True, timeout=TOOL_TIMEOUT_SECONDS)

        return result.stdout

    except subprocess.TimeoutExpired:

        logger.error("Timeout lors de l'exécution de %s", cmd)

        raise

    except subprocess.CalledProcessError as e:

        logger.error("Échec de %s : %s", cmd, e.stderr)

        raise



app_py = Path(WORKDIR) / "app.py"

app_py.write_text(

    "def add(a, b):\n"

    "    return a + b\n\n"

    "def divide(a, b):\n"

    "    return a / b\n",

    encoding="utf-8",

)

test_py = Path(WORKDIR) / "test_app.py"

test_py.write_text(

    "from app import add, divide\n\n"

    "def test_add():\n"

    "    assert add(2, 3) == 5\n\n"

    "def test_divide():\n"

    "    assert divide(10, 2) == 5\n",

    encoding="utf-8",

)



if not (Path(WORKDIR) / ".git").exists():

    run(["git", "init"])

    run(["git", "config", "user.email", "demo@example.com"])

    run(["git", "config", "user.name", "Demo"])

run(["git", "add", "."])

try:

    run(["git", "commit", "-m", "Initial commit: app.py + tests"])

except subprocess.CalledProcessError:

    logger.info("Rien à committer (déjà à jour).")



print("Dépôt de démonstration prêt dans", WORKDIR)

## 1.3 Choisir des serveurs MCP tiers

**Exigence :** au moins deux serveurs tiers. On retient :
1. `@modelcontextprotocol/server-filesystem` (npm, officiel) — lecture/écriture de fichiers.
2. `mcp-server-git` (PyPI, officiel) — statut, historique, diffs Git.

La liste complète est disponible sur [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers).

## 1.4 Se connecter aux serveurs (transport stdio) avec gestion d'erreurs

L'agent lance ces serveurs comme sous-processus stdio. On enveloppe la récupération des outils dans un `try/except` pour échouer proprement (et journaliser) si un serveur ne démarre pas.

In [ ]:
import asyncio

import nest_asyncio

nest_asyncio.apply()



from langchain_mcp_adapters.client import MultiServerMCPClient



mcp_connections = {

    "filesystem": {

        "transport": "stdio",

        "command": "npx",

        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],

    },

    "git": {

        "transport": "stdio",

        "command": "python",

        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],

    },

}



async def get_tools_safe(client, label=""):

    """Récupère les outils MCP avec timeout et gestion d'erreur explicite."""

    try:

        tools = await asyncio.wait_for(client.get_tools(), timeout=TOOL_TIMEOUT_SECONDS)

        logger.info("%s outils chargés depuis %s", len(tools), label or "le client MCP")

        return tools

    except asyncio.TimeoutError:

        logger.error("Timeout lors du chargement des outils MCP (%s)", label)

        raise

    except Exception as e:

        logger.error("Erreur lors du chargement des outils MCP (%s) : %s", label, e)

        raise



client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

tools_day1 = asyncio.get_event_loop().run_until_complete(get_tools_safe(client, "filesystem+git"))



print("Outils disponibles (Jour 1) :")

for t in tools_day1:

    print(" -", t.name)

## 1.5 Construire l'agent Jour 1 (orchestration flexible)

L'agent reçoit tous les outils des deux serveurs tiers. C'est le LLM qui choisit, tour après tour, l'ordre des appels — aucun flux n'est codé en dur.

In [ ]:
from langgraph.prebuilt import create_react_agent



SYSTEM_PROMPT_DAY1 = (

    "Tu es un assistant de développement. Tu disposes d'outils filesystem_* (lire/écrire/lister des "

    "fichiers) et git_* (statut, historique, diff). Choisis toi-même les outils pertinents et leur "

    "ordre pour répondre à la demande. Réponds en français, de façon concise et précise."

)



agent_day1 = create_react_agent(llm, tools_day1, prompt=SYSTEM_PROMPT_DAY1)



async def ask_agent(agent, question: str):

    """Invoque l'agent avec gestion d'erreur/timeout et journalisation."""

    logger.info("Question : %s", question)

    try:

        result = await asyncio.wait_for(

            agent.ainvoke({"messages": [{"role": "user", "content": question}]}),

            timeout=TOOL_TIMEOUT_SECONDS * 4,

        )

        answer = result["messages"][-1].content

        logger.info("Réponse générée (%d caractères)", len(answer))

        print(answer)

        return answer

    except asyncio.TimeoutError:

        logger.error("Timeout global de l'agent pour la question : %s", question)

        print("⚠️ L'agent a dépassé le délai imparti. Réessayez ou augmentez TOOL_TIMEOUT_SECONDS.")

    except Exception as e:

        logger.error("Erreur agent : %s", e)

        print(f"⚠️ Une erreur est survenue : {e}")

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(

    ask_agent(agent_day1, "Liste les fichiers du répertoire de travail, puis donne-moi le message du dernier commit git.")

)

### ✅ Bilan Jour 1
- Deux serveurs MCP tiers connectés (`filesystem`, `git`).
- Orchestration confiée entièrement au LLM (`create_react_agent`).
- Erreurs et timeouts journalisés dans `mcp_agent.log`.

---
# 🗓️ Jour 2 — Construire son propre serveur MCP

**Objectif du jour :** écrire un serveur MCP personnalisé (FastMCP) exposant un **wrapper d'exécution de tests/linter**, l'ajouter au pipeline du Jour 1, puis exposer le tout via une interface Streamlit.

## 2.1 Implémenter le serveur MCP personnalisé

Outils exposés par `dev_ops` :
- `run_tests(test_path)` — exécute `pytest` sur un chemin donné.
- `run_linter(path)` — exécute `ruff`/`flake8` (si disponible) sur un chemin donné.
- `check_syntax(path)` — compile un fichier Python pour détecter une erreur de syntaxe rapidement.

Chaque outil gère explicitement les erreurs courantes : chemin inexistant, timeout du sous-processus, outil externe absent — et renvoie un résultat structuré plutôt que de faire planter le serveur.

In [ ]:
%pip install -qU "fastmcp>=2.0.0" pytest ruff

In [ ]:
from pathlib import Path

import textwrap



server_path = Path("custom_mcp_server.py").resolve()

server_path.write_text(textwrap.dedent('''

    from fastmcp import FastMCP

    from typing import Dict

    import subprocess

    import os

    import py_compile



    mcp = FastMCP(name="dev_ops")



    TIMEOUT = int(os.environ.get("TOOL_TIMEOUT_SECONDS", "30"))



    def _run_subprocess(cmd, cwd) -> Dict[str, object]:

        """Execute une commande avec gestion propre des erreurs/timeouts."""

        try:

            result = subprocess.run(

                cmd, cwd=cwd, capture_output=True, text=True, timeout=TIMEOUT

            )

            return {

                "ok": result.returncode == 0,

                "returncode": result.returncode,

                "stdout": result.stdout[-4000:],

                "stderr": result.stderr[-4000:],

            }

        except FileNotFoundError:

            return {"ok": False, "error": f"Commande introuvable: {cmd[0]}"}

        except subprocess.TimeoutExpired:

            return {"ok": False, "error": f"Timeout depasse ({TIMEOUT}s) pour: {\' \'.join(cmd)}"}



    @mcp.tool

    def ping() -> str:

        """Health check tool."""

        return "pong"



    @mcp.tool

    def run_tests(test_path: str, cwd: str = ".") -> Dict[str, object]:

        """Execute pytest sur test_path (fichier ou dossier) et retourne le resultat structure."""

        if not os.path.exists(os.path.join(cwd, test_path)):

            return {"ok": False, "error": f"Chemin introuvable: {test_path}"}

        return _run_subprocess(["python", "-m", "pytest", test_path, "-q"], cwd=cwd)



    @mcp.tool

    def run_linter(path: str, cwd: str = ".") -> Dict[str, object]:

        """Execute ruff sur path et retourne le resultat structure."""

        if not os.path.exists(os.path.join(cwd, path)):

            return {"ok": False, "error": f"Chemin introuvable: {path}"}

        return _run_subprocess(["python", "-m", "ruff", "check", path], cwd=cwd)



    @mcp.tool

    def check_syntax(path: str, cwd: str = ".") -> Dict[str, object]:

        """Verifie rapidement la syntaxe d'un fichier Python."""

        full_path = os.path.join(cwd, path)

        if not os.path.exists(full_path):

            return {"ok": False, "error": f"Chemin introuvable: {path}"}

        try:

            py_compile.compile(full_path, doraise=True)

            return {"ok": True, "message": "Syntaxe valide"}

        except py_compile.PyCompileError as e:

            return {"ok": False, "error": str(e)}



    if __name__ == "__main__":

        mcp.run(transport="stdio")

'''), encoding="utf-8")



print("Serveur personnalisé écrit :", server_path)

## 2.2 Ajouter le serveur personnalisé au pipeline du Jour 1

In [ ]:
mcp_connections["dev_ops"] = {

    "transport": "stdio",

    "command": "python",

    "args": [str(server_path)],

    "env": {"TOOL_TIMEOUT_SECONDS": str(TOOL_TIMEOUT_SECONDS)},

}



client_full = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

tools_day2 = asyncio.get_event_loop().run_until_complete(get_tools_safe(client_full, "filesystem+git+dev_ops"))



print("Nombre total d'outils :", len(tools_day2))

for t in tools_day2:

    print(" -", t.name)

## 2.3 Reconstruire l'agent avec l'ensemble des serveurs

In [ ]:
SYSTEM_PROMPT_DAY2 = (

    "Tu es un assistant de développement complet. Tu disposes de trois familles d'outils :\n"

    "- filesystem_* : lire/écrire/lister des fichiers.\n"

    "- git_* : statut, historique, diff du dépôt.\n"

    "- dev_ops_* : run_tests, run_linter, check_syntax sur le code du dépôt.\n"

    "Choisis toi-même les outils pertinents et leur ordre pour répondre à la demande. "

    "Si un outil renvoie une erreur, explique-la clairement à l'utilisateur au lieu de l'ignorer. "

    "Réponds en français, de façon concise."

)



agent_full = create_react_agent(llm, tools_day2, prompt=SYSTEM_PROMPT_DAY2)

print("Agent complet (Jour 1 + Jour 2) prêt avec", len(tools_day2), "outils.")

## 2.4 Démonstration de bout en bout

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(

    ask_agent(agent_full, "Exécute les tests du projet (test_app.py), lance le linter sur app.py, "

                          "et résume-moi les résultats en une phrase par outil.")

)

In [ ]:
# Exemple de robustesse : chemin de test inexistant -> l'agent doit gérer l'erreur proprement

_ = asyncio.get_event_loop().run_until_complete(

    ask_agent(agent_full, "Exécute les tests du fichier fichier_qui_n_existe_pas.py et dis-moi ce qui se passe.")

)

---
## 3. Interface utilisateur Streamlit

On enveloppe l'agent complet dans une petite application Streamlit (`app_streamlit.py`), pour un usage interactif hors notebook. La configuration (backend LLM, timeout, répertoire de travail) reste pilotée par le `.env`.

In [ ]:
%%writefile app_streamlit.py

import asyncio

import os



import streamlit as st

from dotenv import load_dotenv

from langchain_mcp_adapters.client import MultiServerMCPClient

from langgraph.prebuilt import create_react_agent



load_dotenv(override=True)



LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()

WORKDIR = os.path.abspath(os.getenv("WORKDIR", "./workspace"))

TOOL_TIMEOUT_SECONDS = int(os.getenv("TOOL_TIMEOUT_SECONDS", "30"))





def get_llm():

    if LLM_BACKEND == "groq":

        from langchain_groq import ChatGroq

        api_key = os.getenv("GROQ_API_KEY")

        if not api_key:

            raise RuntimeError("GROQ_API_KEY manquante dans .env")

        return ChatGroq(model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"), api_key=api_key, temperature=0)

    from langchain_ollama import ChatOllama

    return ChatOllama(

        model=os.getenv("OLLAMA_MODEL", "llama3.1"),

        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),

        temperature=0,

    )





@st.cache_resource(show_spinner="Connexion aux serveurs MCP...")

def build_agent():

    mcp_connections = {

        "filesystem": {

            "transport": "stdio",

            "command": "npx",

            "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],

        },

        "git": {

            "transport": "stdio",

            "command": "python",

            "args": ["-m", "mcp_server_git", "--repository", WORKDIR],

        },

        "dev_ops": {

            "transport": "stdio",

            "command": "python",

            "args": [os.path.abspath("custom_mcp_server.py")],

            "env": {"TOOL_TIMEOUT_SECONDS": str(TOOL_TIMEOUT_SECONDS)},

        },

    }

    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

    tools = asyncio.run(client.get_tools())

    prompt = (

        "Tu es un assistant de développement avec des outils filesystem_*, git_* et dev_ops_*. "

        "Choisis toi-même les outils pertinents. Réponds en français, de façon concise. "

        "Si un outil échoue, explique l'erreur clairement."

    )

    return create_react_agent(get_llm(), tools, prompt=prompt)





st.set_page_config(page_title="Assistant de développement MCP", page_icon="🛠️")

st.title("🛠️ Assistant de développement (MCP + Agents)")

st.caption(f"Backend LLM : {LLM_BACKEND} · Répertoire de travail : {WORKDIR}")



if "history" not in st.session_state:

    st.session_state.history = []



try:

    agent = build_agent()

except Exception as e:

    st.error(f"Impossible d'initialiser l'agent : {e}")

    st.stop()



for role, content in st.session_state.history:

    with st.chat_message(role):

        st.markdown(content)



if question := st.chat_input("Demandez quelque chose (ex: exécute les tests et le linter)"):

    st.session_state.history.append(("user", question))

    with st.chat_message("user"):

        st.markdown(question)



    with st.chat_message("assistant"):

        with st.spinner("Réflexion et appel des outils..."):

            try:

                result = asyncio.run(

                    asyncio.wait_for(

                        agent.ainvoke({"messages": [{"role": "user", "content": question}]}),

                        timeout=TOOL_TIMEOUT_SECONDS * 4,

                    )

                )

                answer = result["messages"][-1].content

            except asyncio.TimeoutError:

                answer = "⚠️ Délai dépassé. Réessayez ou augmentez TOOL_TIMEOUT_SECONDS dans .env."

            except Exception as e:

                answer = f"⚠️ Erreur : {e}"

        st.markdown(answer)

    st.session_state.history.append(("assistant", answer))

Pour lancer l'interface (en dehors du notebook, dans un terminal, à la racine du projet) :

```bash
streamlit run app_streamlit.py
```

---
## 4. Reproductibilité : `requirements.txt` et démarrage

Objectif : `clone + install + run` sans étape cachée.

In [ ]:
%%writefile requirements.txt

langchain>=0.3

langgraph>=0.2

langchain-mcp-adapters==0.2.1

langchain-groq>=0.2

langchain-ollama>=0.2

mcp-server-git

fastmcp>=2.0.0

python-dotenv

nest_asyncio

streamlit

pytest

ruff

In [ ]:
%%writefile README.md

# Assistant de développement — MCP + Agents



## Démarrage rapide



```bash

git clone <url-du-depot>

cd <depot>

python -m venv .venv && source .venv/bin/activate

pip install -r requirements.txt

cp .env.example .env   # puis éditer .env (backend LLM, clé API...)

streamlit run app_streamlit.py

```



## Contenu

- `custom_mcp_server.py` : serveur MCP personnalisé (run_tests, run_linter, check_syntax).

- `app_streamlit.py` : interface de chat sur l'agent complet.

- `.env.example` : variables d'environnement à copier vers `.env`.

- `mcp_agent.log` : journal des appels d'outils et des erreurs.



## Backends LLM supportés

- **Ollama** (local, gratuit) : `LLM_BACKEND=ollama`, nécessite `ollama serve` lancé localement.

- **GroqCloud** (API) : `LLM_BACKEND=groq`, nécessite `GROQ_API_KEY`.

---
## ✅ Bilan final

- **Jour 1** : deux serveurs MCP tiers (`filesystem`, `git`) connectés et orchestrés par un LLM (Ollama ou Groq), avec timeouts et journalisation.
- **Jour 2** : serveur MCP personnalisé `dev_ops` (tests, lint, vérification de syntaxe) ajouté au même pipeline, sans changer l'architecture d'orchestration.
- **Robustesse** : chaque outil et chaque appel d'agent gèrent proprement timeout, entrée invalide ou commande absente, avec journalisation dans `mcp_agent.log`.
- **Reproductibilité** : `requirements.txt`, `.env.example`, `README.md` et `app_streamlit.py` permettent un démarrage `clone + install + run`.